# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** · Meridiano Imobiliária

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu a base **como ela
chega**: 2.930 registros, 82 colunas, 27 delas com célula vazia, escalas de qualidade
guardadas como texto, dois anos impossíveis e cinco imóveis que o próprio autor da base
recomenda descartar. Este notebook é o passo seguinte: **aplicar as correções** e gravar
a base curada que todas as fases posteriores consomem.

A regra deste notebook é uma só:

> **Nenhum dado é alterado em silêncio.** Cada ajuste declara o *motivo*, mostra a
> *evidência numérica* que o justifica e registra *quantos registros* foram tocados.


In [50]:
# Importações
import pandas as pd
import numpy as np

In [51]:
# Carregando a base de dados
df = pd.read_csv("../data/processed/ames_housing_columns_selected.csv", sep=';')

In [52]:
# Ajustando o ano_garagem (erro de digitação)
df.loc[df["ano_garagem"] == 2207, "ano_garagem"] = 2007

# Ano reforma inferior ao ano de construção
mascara = df["ano_reforma"] < df["ano_construcao"]
df.loc[mascara, "ano_reforma"] = df.loc[mascara, "ano_construcao"]

# Porão existe, mas exposição vazia; imputar pela moda
tem_porao = df["qualidade_porao"].notna()
df.loc[tem_porao & df["exposicao_porao"].isna(), "exposicao_porao"] = df["exposicao_porao"].mode()[0]

# Porão existe, mas acabamento_porao_2 vazio com área; imputar pela moda das casas com área
com_area = df["area_porao_acabada_2"].gt(0)
moda_acab_2 = df.loc[com_area & df["acabamento_porao_2"].ne("Sem acabamento"), "acabamento_porao_2"].mode()[0]
df.loc[com_area & df["acabamento_porao_2"].isna(), "acabamento_porao_2"] = moda_acab_2

# Alinhando acabamento_porao_2 à área
df.loc[com_area & df["acabamento_porao_2"].eq("Sem acabamento"), "acabamento_porao_2"] = moda_acab_2
df.loc[df["area_porao_acabada_2"].eq(0) & df["acabamento_porao_2"].notna(), "acabamento_porao_2"] = "Sem acabamento"

# Garagem separada com características vazias; imputar pela moda das garagens separadas
separada = df["tipo_garagem"].eq("Separada da casa")
for col in ["acabamento_garagem", "qualidade_garagem", "conservacao_garagem"]:
    df.loc[separada & df[col].isna() & df["area_garagem"].gt(0), col] = df.loc[separada, col].mode()[0]

# Garagem separada sem nenhuma informação; tratar como sem garagem
df.loc[separada & df["area_garagem"].isna(), "tipo_garagem"] = np.nan

# Alvenaria ausente; tratar como sem alvenaria
df["tipo_alvenaria_aparente"] = df["tipo_alvenaria_aparente"].fillna("Sem alvenaria aparente")

# Alinhando tipo e área da alvenaria (área relevante sem tipo recebe a moda; área de 1 pé² vira 0)
sem_alvenaria = df["tipo_alvenaria_aparente"].eq("Sem alvenaria aparente")
moda_alvenaria = df.loc[~sem_alvenaria, "tipo_alvenaria_aparente"].mode()[0]
df.loc[sem_alvenaria & df["area_alvenaria_aparente"].gt(1), "tipo_alvenaria_aparente"] = moda_alvenaria
df.loc[df["tipo_alvenaria_aparente"].eq("Sem alvenaria aparente"), "area_alvenaria_aparente"] = 0
df.loc[df["area_alvenaria_aparente"].eq(0), "tipo_alvenaria_aparente"] = "Sem alvenaria aparente"

# Ausentes estruturais (o item não existe)
ausencia = {
    "Sem porão": ["qualidade_porao", "conservacao_porao", "exposicao_porao", "acabamento_porao_1", "acabamento_porao_2"],
    "Sem garagem": ["tipo_garagem", "acabamento_garagem", "qualidade_garagem", "conservacao_garagem"],
    "Sem lareira": ["qualidade_lareira"],
    "Sem piscina": ["qualidade_piscina"],
    "Sem cerca": ["tipo_cerca"],
    "Sem viela": ["tipo_viela"],
    "Nenhum": ["item_extra"],
}
for rotulo, colunas in ausencia.items():
    df[colunas] = df[colunas].fillna(rotulo)

# Áreas, vagas e banheiros ausentes de porão e garagem são zero
cols_zero = ["area_porao", "area_porao_acabada_1", "area_porao_acabada_2", "area_porao_bruta",
             "banheiros_completos_porao", "lavabos_porao", "vagas_garagem", "area_garagem"]
df[cols_zero] = df[cols_zero].fillna(0)

# Ano da garagem ausente (sem garagem); usar o ano de construção
df["ano_garagem"] = df["ano_garagem"].fillna(df["ano_construcao"]).astype("Int64")

# Testada do lote ausente; imputar pela mediana do bairro
df["testada_lote"] = df["testada_lote"].fillna(df.groupby("bairro")["testada_lote"].transform("median"))
df["testada_lote"] = df["testada_lote"].fillna(df["testada_lote"].median())

# Sistema elétrico ausente; imputar pela moda
df["sistema_eletrico"] = df["sistema_eletrico"].fillna(df["sistema_eletrico"].mode()[0])

# Removendo as 3 vendas parciais com área acima de 4.000 pés² e preço incompatível
df = df[~(df["area_habitavel"].gt(4000) & df["condicao_venda"].eq("Parcial"))]

# Ausentes restantes por coluna
nulos = df.isna().mean().mul(100).round(2)
print(nulos[nulos > 0])

# Colunas com mais de 5% de ausentes são removidas
colunas_removidas = nulos[nulos > 5].index.tolist()
df = df.drop(columns=colunas_removidas)
print(f"Colunas removidas: {colunas_removidas}")

# Registros com mais de 5% de ausentes são removidos
antes = len(df)
df = df[df.isna().mean(axis=1).le(0.05)]
print(f"{antes - len(df)} registros removidos")

# Demais ausentes: imputação pela mediana (numéricas) ou moda (categóricas)
for col in df.columns[df.isna().any()]:
    valor = df[col].median() if pd.api.types.is_numeric_dtype(df[col]) else df[col].mode()[0]
    print(f"{col}: {df[col].isna().sum()} imputados com {valor}")
    df[col] = df[col].fillna(valor)

# Idade do imóvel na venda (vendas na planta dariam idade negativa)
df["idade_imovel"] = (df["ano_venda"] - df["ano_construcao"]).clip(lower=0)

# Anos desde a última reforma
df["anos_desde_reforma"] = (df["ano_venda"] - df["ano_reforma"]).clip(lower=0)

# Flag de reforma (1950 é o piso do cadastro: casas antigas com reforma em 1950 não contam)
df["foi_reformada"] = (df["ano_reforma"] > df["ano_construcao"]) & (df["ano_reforma"] > 1950)
df["foi_reformada"] = df["foi_reformada"].astype(int)

# Área total de varandas
df["area_varandas"] = df[["area_varanda_aberta", "area_varanda_fechada", "area_varanda_3estacoes", "area_varanda_tela"]].sum(axis=1)

# Total de banheiros (lavabo conta meio banheiro)
df["banheiros_total"] = (df["banheiros_completos"] + df["banheiros_completos_porao"]
                         + 0.5 * (df["lavabos"] + df["lavabos_porao"]))

# Flags de existência
df["tem_porao"] = df["area_porao"].gt(0).astype(int)
df["tem_garagem"] = df["area_garagem"].gt(0).astype(int)
df["tem_lareira"] = df["lareiras"].gt(0).astype(int)
df["tem_segundo_pavimento"] = df["area_pavimento_superior"].gt(0).astype(int)

# Salvando a base tratada
df.to_csv("../data/processed/ames_housing_tratado.csv", sep=';', index=False)

infraestrutura            0.10
qualidade_externa        61.46
conservacao_externa      86.98
qualidade_aquecimento    29.52
qualidade_cozinha        51.04
funcionalidade            6.90
acesso_pavimentado        7.38
dtype: float64
Colunas removidas: ['qualidade_externa', 'conservacao_externa', 'qualidade_aquecimento', 'qualidade_cozinha', 'funcionalidade', 'acesso_pavimentado']
0 registros removidos
infraestrutura: 3 imputados com Todos os serviços
